# Packages

In [ ]:
import glob
import argparse
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import oggm
from oggm import utils

In [ ]:
import warnings

from sklearn.model_selection import train_test_split

import xgboost as xgb

from sklearn.metrics import confusion_matrix

from sklearn.metrics import log_loss

from sklearn.metrics import accuracy_score

from sklearn.metrics import mean_squared_error

from sklearn.metrics import roc_curve

from sklearn.metrics import auc

#import lightgbm as lgb

#from lightgbm import early_stopping

#import time

from sklearn.neural_network import MLPRegressor

from xgboost import XGBRegressor

from sklearn.model_selection import RandomizedSearchCV

from scipy.stats import randint, uniform

from sklearn.ensemble import RandomForestRegressor

from sklearn.datasets import make_regression

from sklearn.metrics import mean_absolute_error

# Load metadata, plotting marginal distributions and cross tabulations

In [ ]:
# Setup oggm: important we want to use oggm's version 62 of all glacier geometries. This command will download them.
utils.get_rgi_dir(version='62')

In [ ]:
path_O1_shp = '/Users/emma/OGGM/rgi/RGIV62/00_rgi62_regions/00_rgi62_O1Regions.shp' # shp file of regional boundaries

# Import 20-bin gridded training dataset
metadata_file = "//Users/emma/aml/GroupProject/metadata19-001.csv"
glathida_rgis = pd.read_csv(metadata_file, low_memory=False)

In [ ]:
glathida_rgis

In [ ]:
# Kald funktionen og gem resultatet
counts = count_na_and_nan(glathida_rgis)

# Udskriv resultaterne
for column, count_dict in counts.items():
    print(f"Column '{column}':")
    print(f"Occurrences of 'na': {count_dict['na_count']}")
    print(f"Occurrences of 'nan': {count_dict['nan_count']}")
    print()

In [ ]:
crosstab6 = pd.crosstab(glathida_rgis['GlaThiDa_ID'], glathida_rgis['GLACIER_NAME'])
print(crosstab6)

In [ ]:
# Brug shape-attributtet til at få antallet af rækker og kolonner
num_rows, num_columns = glathida_rgis.shape

print(f"Antal rækker i datasættet: {num_rows}")
print(f"Antal kolonner i datasættet: {num_columns}")

# Beregn antallet af manglende værdier i hver række og kolonne
num_missing_rows = glathida_rgis.isna().any(axis=1).sum()
num_missing_columns = glathida_rgis.isna().any(axis=0).sum()

print(f"Antal rækker med manglende værdier: {num_missing_rows}")
print(f"Antal kolonner med manglende værdier: {num_missing_columns}")

In [ ]:
len(pd.unique(glathida_rgis['GlaThiDa_ID']))

In [ ]:
for col in glathida_rgis.columns:
    print(f"{col}: {glathida_rgis[col].dtype}")

In [ ]:
# Plot de marginale fordelinger af hver variabel
fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(15, 10))

columns_to_plot = ['GlaThiDa_ID', 'POLITICAL_UNIT',
                   'SURVEY_DATE','PROFILE_ID', 'POINT_ID', 
                   'POINT_LAT','POINT_LON', 'ELEVATION',
                   'THICKNESS','THICKNESS_UNCERTAINTY', "REMARKS"]

for i, col in enumerate(columns_to_plot):
    ax = axes[i // 3, i % 3]
    glathida_rgis[col].hist(ax=ax, bins=20, color='skyblue', edgecolor='black')
    ax.set_title(f'Marginal fordeling af {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Antal observationer')
    ax.grid(False)

# Juster layout
plt.tight_layout()
plt.show()

In [ ]:
# Plot de marginale fordelinger af hver variabel
fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(15, 10))

columns_to_plot = ['THICKNESS_UNCERTAINTY', 'DATA_FLAG',
                   'REMARKS','RGI', 'RGIId', 
                   'Area','Zmin', 'Zmax',
                   'Zmed','Slope', "Lmax", 'Form']

for i, col in enumerate(columns_to_plot):
    ax = axes[i // 3, i % 3]
    glathida_rgis[col].hist(ax=ax, bins=20, color='skyblue', edgecolor='black')
    ax.set_title(f'Marginal fordeling af {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Antal observationer')
    ax.grid(False)

# Juster layout
plt.tight_layout()
plt.show()

In [ ]:
# Plot de marginale fordelinger af hver variabel
fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(15, 10))

columns_to_plot = ['TermType', 'Aspect',
                   'dmdtda_hugo','elevation', 'slope_lat', 
                   'slope_lon','slope_lat_gf50', 'slope_lon_gf50',
                   'slope_lat_gf100','slope_lon_gf100', "slope_lat_gf150", 'slope_lon_gf150']

for i, col in enumerate(columns_to_plot):
    ax = axes[i // 3, i % 3]
    glathida_rgis[col].hist(ax=ax, bins=20, color='skyblue', edgecolor='black')
    ax.set_title(f'Marginal fordeling af {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Antal observationer')
    ax.grid(False)

# Juster layout
plt.tight_layout()
plt.show()

In [ ]:
# Plot de marginale fordelinger af hver variabel
fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(15, 10))

columns_to_plot = ['slope_lat_gf300', 'slope_lon_gf300',
                   'slope_lat_gf450','slope_lon_gf450', 'slope_lat_gfa', 
                   'slope_lon_gfa','curv_50', 'curv_300',
                   'curv_gfa','aspect_50', "aspect_300", 'aspect_gfa']

for i, col in enumerate(columns_to_plot):
    ax = axes[i // 3, i % 3]
    glathida_rgis[col].hist(ax=ax, bins=20, color='skyblue', edgecolor='black')
    ax.set_title(f'Marginal fordeling af {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Antal observationer')
    ax.grid(False)

# Juster layout
plt.tight_layout()
plt.show()

In [ ]:
# Plot de marginale fordelinger af hver variabel
fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(15, 10))

columns_to_plot = ['smb', 'ith_m',
                   'vx','vy', 'vx_gf50', 
                   'vx_gf100','vx_gf150', 'vx_gf300',
                   'vx_gf450','vx_gfa', "vy_gf50", 'vy_gf100']

for i, col in enumerate(columns_to_plot):
    ax = axes[i // 3, i % 3]
    glathida_rgis[col].hist(ax=ax, bins=20, color='skyblue', edgecolor='black')
    ax.set_title(f'Marginal fordeling af {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Antal observationer')
    ax.grid(False)

# Juster layout
plt.tight_layout()
plt.show()

In [ ]:
# Beregn 'slope_total' for hver række
glathida_rgis['slope_total'] = np.sqrt(glathida_rgis['slope_lat']**2 + glathida_rgis['slope_lon']**2)

glathida_rgis['slope_total_gfa'] = np.sqrt(glathida_rgis['slope_lat_gfa']**2 + glathida_rgis['slope_lon_gfa']**2)

# Gem det opdaterede datasæt tilbage til en ny CSV-fil
#df.to_csv('glathida_rgis_edited_with_slope_total.csv', index=False)

#print("Beregningen af 'slope_total' er fuldført, og resultatet er gemt i 'glathida_rgis_edited_with_slope_total.csv'")

In [ ]:
# Plot de marginale fordelinger af hver variabel
fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(15, 10))

columns_to_plot = ['vy_gf150', 'vy_gf300',
                   'vy_gf450','vy_gfa', 'dvx_dx', 
                   'dvx_dy','dvy_dx', 'dvy_dy',
                   'dist_from_border_km_geom','ith_f', "slope_total", 'slope_total_gfa']

for i, col in enumerate(columns_to_plot):
    ax = axes[i // 3, i % 3]
    glathida_rgis[col].hist(ax=ax, bins=20, color='skyblue', edgecolor='black')
    ax.set_title(f'Marginal fordeling af {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Antal observationer')
    ax.grid(False)

# Juster layout
plt.tight_layout()
plt.show()

In [ ]:
pd.unique(glathida_rgis['THICKNESS_UNCERTAINTY'])

In [ ]:
pd.unique(glathida_rgis["GlaThiDa_ID"])

In [ ]:
pd.unique(glathida_rgis["POLITICAL_UNIT"])

In [ ]:
pd.unique(glathida_rgis["PROFILE_ID"])

In [ ]:
pd.unique(glathida_rgis["REMARKS"])

In [ ]:
len(pd.unique(glathida_rgis["GLACIER_NAME"]))

In [ ]:
pd.unique(glathida_rgis["THICKNESS_UNCERTAINTY"])

In [ ]:
#pd.unique(glathida_rgis["REMARKS"])
pd.unique(glathida_rgis["DATA_FLAG"])

In [ ]:
# Erstat alle NaN-værdier i kolonnen 'DATA_FLAG' med 0
glathida_rgis['DATA_FLAG'].fillna(0, inplace=True)

In [ ]:
# Tæl antallet af forekomster af hver unik værdi i kolonnen 'DATA_FLAG'
flag_counts = glathida_rgis['DATA_FLAG'].value_counts()

# Udskriv antallet af 0'er og 1'ere
print("Antal 0'er:", flag_counts.get(0, 0))
print("Antal 1'ere:", flag_counts.get(1, 0))

In [ ]:
# Lav et udtræk af rækkerne hvor 'DATA_FLAG' er lig med 1
data_flag_1 = glathida_rgis[glathida_rgis['DATA_FLAG'] == 1]

# Udskriv det resulterende udtræk
print(data_flag_1)

In [ ]:
# Juster displayindstillinger for at vise alle kolonner
pd.set_option('display.max_columns', None)

# Antag at 'glathida_rgis' er dit DataFrame

# Lav et udtræk af rækkerne hvor 'DATA_FLAG' er lig med 1
data_flag_1 = glathida_rgis[glathida_rgis['DATA_FLAG'] == 1]

# Udskriv det resulterende udtræk
print(data_flag_1)

In [ ]:
# Fjern rækkerne hvor 'DATA_FLAG' er lig med 1
glathida_rgis_edited = glathida_rgis[glathida_rgis['DATA_FLAG'] != 1]

# Fjern kolonnen "DATA_FLAG"
glathida_rgis_edited.drop("DATA_FLAG", axis=1, inplace=True)

In [ ]:
# Brug shape-attributtet til at få antallet af rækker og kolonner
num_rows, num_columns = glathida_rgis_edited.shape

print(f"Antal rækker i datasættet: {num_rows}")
print(f"Antal kolonner i datasættet: {num_columns}")

# FØR fjernelse af thickness=0 : 3.854.278 og 67
# EFTER: 3.679.350

In [ ]:
# Opret et subset, hvor 'THICKNESS' er forskellig fra 0
glathida_rgis_edited = glathida_rgis_edited[glathida_rgis_edited['THICKNESS'] != 0]

# Udskriv det resulterende subset for at kontrollere
print(glathida_rgis_edited)

# Cleaning the data

## Checker for missing values

In [ ]:
import pandas as pd
import re

# Antag at "glathida_rgis.csv" er dit datasæt
# Læs datasættet ind i en pandas DataFrame

def count_na_and_nan(dataframe):
    counts = {}
    for column in dataframe.columns:
        # Konverter kolonneværdier til strenge, hvis de ikke allerede er strenge
        column_values_as_strings = dataframe[column].astype(str)
        
        # Saml alle værdier fra kolonnen til én lang streng
        column_text = " ".join(column_values_as_strings)
        
        # Find forekomster af 'na' og 'nan' i kolonnen
        na_matches = re.findall(r'\bna\b', column_text)
        nan_matches = re.findall(r'\bnan\b', column_text)
        
        # Tæl forekomsterne
        na_count = len(na_matches)
        nan_count = len(nan_matches)
        
        # Gem tællingerne for denne kolonne i dictionaryen
        counts[column] = {'na_count': na_count, 'nan_count': nan_count}
        
    return counts

# Kald funktionen og gem resultatet
counts = count_na_and_nan(glathida_rgis_edited)

# Udskriv resultaterne
for column, count_dict in counts.items():
    print(f"Column '{column}':")
    print(f"Occurrences of 'na': {count_dict['na_count']}")
    print(f"Occurrences of 'nan': {count_dict['nan_count']}")
    print()


In [ ]:
# Beregn andelen af NaN-værdier eller "na" i hver kolonne
na_percentage = glathida_rgis_edited.isna().sum() / len(glathida_rgis_edited)

# Find kolonner, hvor andelen af NaN-værdier eller "na" er mere end 25%
columns_to_keep = na_percentage[na_percentage > 0.25].index

# Opret et subsæt af data ved kun at vælge de ønskede kolonner
subset_data = glathida_rgis_edited[columns_to_keep]

# Udskriv de valgte kolonner
print("Valgte kolonner med mere end 10% NaN eller 'na' værdier:")
print(subset_data.head())


In [ ]:
# Beregn andelen af NaN-værdier i hver række
na_percentage = glathida_rgis_edited.isna().sum(axis=1) / glathida_rgis_edited.shape[1]

# Find rækker, hvor andelen af NaN-værdier er mere end 25%
rows_to_keep = na_percentage[na_percentage >= 0.25].index

# Opret et subset af data ved kun at vælge de ønskede rækker
subset_data = glathida_rgis_edited.loc[rows_to_keep]

# Udskriv de valgte rækker
print("Rækker med mere end 2.5% NaN-værdier:")
print(subset_data)

(34662 / 3854279)*100 = 0.9%  Dvs efter fjernelse af rækker er der stadig 3,819,617 antal rækker

In [ ]:
# Beregn andelen af NaN-værdier i hver række
na_percentage = glathida_rgis.isna().sum(axis=1) / glathida_rgis.shape[1]

# Find rækker, hvor andelen af NaN-værdier er mere end 25%
rows_to_keep = na_percentage[na_percentage >= 0.5].index

# Opret et subsæt af data ved kun at vælge de ønskede rækker
subset = glathida_rgis.loc[rows_to_keep]

# Udskriv de valgte rækker
print("Valgte rækker med mere end 25% NaN-værdier:")
print(subset)

In [ ]:
# Lag en ny kolonne som indikerer om 'floor' er NaN
glathida_rgis_edited['is_na_glathida_id'] = glathida_rgis_edited['GlaThiDa_ID'].isna()
glathida_rgis_edited['is_na_remarks'] = glathida_rgis_edited['REMARKS'].isna()
glathida_rgis_edited['is_na_profile_id'] = glathida_rgis_edited['PROFILE_ID'].isna()
glathida_rgis_edited['is_na_glacier_name'] = glathida_rgis_edited['GLACIER_NAME'].isna()
glathida_rgis_edited['is_na_thickness'] = glathida_rgis_edited['THICKNESS_UNCERTAINTY'].isna()
glathida_rgis_edited['is_na_elevation'] = glathida_rgis_edited['ELEVATION'].isna()

glathida_rgis_edited['is_na_slope'] = glathida_rgis_edited['Slope'].isna()
glathida_rgis_edited['is_na_form'] = glathida_rgis_edited['Form'].isna()
glathida_rgis_edited['is_na_aspect'] = glathida_rgis_edited['Aspect'].isna()
glathida_rgis_edited['is_na_dist_from_border_km_geom'] = glathida_rgis_edited['dist_from_border_km_geom'].isna()

In [ ]:
crosstab1 = pd.crosstab(glathida_rgis['THICKNESS_UNCERTAINTY'], glathida_rgis['THICKNESS'],)
print(crosstab1)

In [ ]:
glathida_rgis['is_na_remarks'] = glathida_rgis['REMARKS'].isna()
crosstab1 = pd.crosstab(glathida_rgis['THICKNESS'], glathida_rgis['is_na_remarks'])
print(crosstab1)

In [ ]:
crosstab1 = pd.crosstab(glathida_rgis['THICKNESS'], glathida_rgis['REMARKS'])
print(crosstab1)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Antagelse: glathida_rgis DataFrame er allerede defineret og indeholder kolonnen 'REMARKS' og 'THICKNESS'.

# Tilføj en ny kolonne for at indikere manglende værdier i 'REMARKS'
glathida_rgis['is_na_remarks'] = glathida_rgis['REMARKS'].isna()

# Beregn crosstab
crosstab1 = pd.crosstab(glathida_rgis['THICKNESS'], glathida_rgis['is_na_remarks'])
print(crosstab1)

# Ekstraher 'True' kolonnen
true_counts = crosstab1[True]

# Lav et histogram
plt.figure(figsize=(10, 6))
true_counts.plot(kind='bar', color='skyblue')
plt.xlabel('THICKNESS')
plt.ylabel('Count of True is_na_remarks')
plt.title('Histogram of True is_na_remarks by THICKNESS')
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Antagelse: glathida_rgis DataFrame er allerede defineret og indeholder kolonnerne 'REMARKS' og 'THICKNESS'.

# Tilføj en ny kolonne for at indikere manglende værdier i 'REMARKS'
glathida_rgis['is_na_remarks'] = glathida_rgis['REMARKS'].isna()

# Beregn crosstab
crosstab1 = pd.crosstab(glathida_rgis['THICKNESS'], glathida_rgis['is_na_remarks'])

# Ekstraher 'True' kolonnen
true_counts = crosstab1[True]

# Beregn fordelingen af 'THICKNESS'
thickness_distribution = glathida_rgis['THICKNESS'].value_counts().sort_index()

# Opret et nyt figur og en ny akse
fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot histogram af True is_na_remarks counts
ax1.bar(true_counts.index, true_counts, color='skyblue', label='Count of True is_na_remarks', alpha=0.6)
ax1.set_xlabel('THICKNESS')
ax1.set_ylabel('Count of True is_na_remarks', color='skyblue')
ax1.tick_params(axis='y', labelcolor='skyblue')

# Opret en anden y-akse for thickness distribution
ax2 = ax1.twinx()
ax2.plot(thickness_distribution.index, thickness_distribution, color='orange', marker='o', label='THICKNESS Distribution')
ax2.set_ylabel('THICKNESS Distribution', color='orange')
ax2.tick_params(axis='y', labelcolor='orange')

# Tilføj en titel og vis plot
fig.suptitle('Comparison of True is_na_remarks Counts and THICKNESS Distribution')
fig.legend(loc='upper right')

plt.show()


In [ ]:
# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab1 = pd.crosstab(glathida_rgis_edited['is_na_glacier_name'], glathida_rgis_edited['is_na_remarks'])
print(crosstab1)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab2 = pd.crosstab(glathida_rgis_edited['is_na_glacier_name'], glathida_rgis_edited['is_na_thickness'])
print(crosstab2)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab4 = pd.crosstab(glathida_rgis_edited['is_na_glacier_name'], glathida_rgis_edited['is_na_elevation'])
print(crosstab4)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab5 = pd.crosstab(glathida_rgis_edited['is_na_glacier_name'], glathida_rgis_edited['is_na_profile_id'])
print(crosstab5)

crosstab6 = pd.crosstab(glathida_rgis_edited['is_na_thickness'], glathida_rgis_edited['is_na_remarks'])
print(crosstab6)

In [ ]:
# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab1 = pd.crosstab(glathida_rgis_edited['is_na_slope'], glathida_rgis_edited['is_na_form'])
print(crosstab1)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab2 = pd.crosstab(glathida_rgis_edited['is_na_slope'], glathida_rgis_edited['is_na_aspect'])
print(crosstab2)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab3 = pd.crosstab(glathida_rgis_edited['is_na_form'], glathida_rgis_edited['is_na_aspect'])
print(crosstab3)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab4 = pd.crosstab(glathida_rgis_edited['is_na_form'], glathida_rgis_edited['is_na_aspect'])
print(crosstab4)

In [ ]:
# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab1 = pd.crosstab(glathida_rgis_edited['is_na_slope'], glathida_rgis_edited['is_na_remarks'])
print(crosstab1)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab2 = pd.crosstab(glathida_rgis_edited['is_na_slope'], glathida_rgis_edited['is_na_thickness'])
print(crosstab2)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab4 = pd.crosstab(glathida_rgis_edited['is_na_slope'], glathida_rgis_edited['is_na_elevation'])
print(crosstab4)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab5 = pd.crosstab(glathida_rgis_edited['is_na_slope'], glathida_rgis_edited['is_na_profile_id'])
print(crosstab5)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab6 = pd.crosstab(glathida_rgis_edited['is_na_dist_from_border_km_geom'], glathida_rgis_edited['is_na_glacier_name'])
print(crosstab6)

In [ ]:
# Opret en krydstabel
crosstab_result = pd.crosstab(glathida_rgis_edited['REMARKS'], glathida_rgis_edited['is_na_thickness'])
crosstab_result = pd.crosstab(glathida_rgis_edited['is_na_remarks'], glathida_rgis_edited['THICK'])

# Udskriv krydstabellen
print(crosstab_result)

In [ ]:
# Opret en krydstabel
crosstab_result = pd.crosstab(glathida_rgis_edited['REMARKS'], glathida_rgis_edited['is_na_thickness'])
crosstab_result = pd.crosstab(glathida_rgis_edited['is_na_remarks'], glathida_rgis_edited['THICK'])

# Udskriv krydstabellen
print(crosstab_result)

In [ ]:
import seaborn as sns
#Lav en kopi af datasættet, hvor NaN-værdier er markeret som True og NA-værdier som False
na_nan_indicator = glathida_rgis.isna()

# Plot en heatmap af manglende værdier og NaN'er
plt.figure(figsize=(10, 6))
sns.heatmap(na_nan_indicator, cmap='viridis', cbar=False)
plt.title('Heatmap af manglende værdier og NaN\'er')
plt.xlabel('Kolonner')
plt.ylabel('Rækker')
plt.show()

# Beregn korrelationskoefficienten mellem NA-værdier og NaN-værdier
correlation = na_nan_indicator.sum(axis=1).corr(na_nan_indicator.sum(axis=0))

print(f"Korrelationskoefficienten mellem NA-værdier og NaN-værdier: {correlation}")

In [ ]:
# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab1 = pd.crosstab(glathida_rgis_edited['is_na_remarks'], glathida_rgis_edited['is_na_glacier_name'])
print(crosstab1)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab2 = pd.crosstab(glathida_rgis_edited['is_na_remarks'], glathida_rgis_edited['is_na_thickness'])
print(crosstab2)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab4 = pd.crosstab(glathida_rgis_edited['is_na_remarks'], glathida_rgis_edited['is_na_elevation'])
print(crosstab4)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab5 = pd.crosstab(glathida_rgis_edited['is_na_remarks'], glathida_rgis_edited['is_na_profile_id'])
print(crosstab5)

In [ ]:
# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab1 = pd.crosstab(subset['is_na_profile_id'], subset['is_na_glacier_name'])
print(crosstab1)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab2 = pd.crosstab(subset['is_na_profile_id'], subset['is_na_thickness'])
print(crosstab2)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab3 = pd.crosstab(subset['is_na_profile_id'], subset['is_na_data_flag'])
print(crosstab3)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab4 = pd.crosstab(subset['is_na_profile_id'], subset['is_na_elevation'])
print(crosstab4)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab5 = pd.crosstab(subset['is_na_profile_id'], subset['is_na_remarks'])
print(crosstab5)

In [ ]:
# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab1 = pd.crosstab(subset['is_na_glacier_name'], subset['is_na_profile_id'])
print(crosstab1)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab2 = pd.crosstab(subset['is_na_glacier_name'], subset['is_na_thickness'])
print(crosstab2)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab3 = pd.crosstab(subset['is_na_glacier_name'], subset['is_na_data_flag'])
print(crosstab3)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab4 = pd.crosstab(subset['is_na_glacier_name'], subset['is_na_elevation'])
print(crosstab4)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab5 = pd.crosstab(subset['is_na_glacier_name'], subset['is_na_remarks'])
print(crosstab5)

In [ ]:
# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab1 = pd.crosstab(subset['is_na_thickness'], subset['is_na_profile_id'])
print(crosstab1)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab2 = pd.crosstab(subset['is_na_thickness'], subset['is_na_glacier_name'])
print(crosstab2)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab3 = pd.crosstab(subset['is_na_thickness'], subset['is_na_data_flag'])
print(crosstab3)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab4 = pd.crosstab(subset['is_na_thickness'], subset['is_na_elevation'])
print(crosstab4)

# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab5 = pd.crosstab(subset['is_na_thickness'], subset['is_na_remarks'])
print(crosstab5)

In [ ]:
# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab = pd.crosstab(glathida_rgis_edited['is_na_profile_id'], glathida_rgis_edited['GLACIER_NAME'])

print(crosstab)

In [ ]:
# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab = pd.crosstab(glathida_rgis_edited['GLACIER_NAME'], glathida_rgis_edited['PROFILE_ID'])

print(crosstab)

In [ ]:
# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab = pd.crosstab(glathida_rgis_edited['is_na_glacier_name'], glathida_rgis_edited['POLITICAL_UNIT'])

print(crosstab)

In [ ]:
# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab = pd.crosstab(glathida_rgis_edited['is_na_profile_id'], glathida_rgis_edited['POLITICAL_UNIT'])

print(crosstab)

In [ ]:
glathida_rgis_edited['is_na_glathida_id'] = glathida_rgis_edited['GlaThiDa_ID'].isna()
glathida_rgis_edited['is_na_servey_date'] = glathida_rgis_edited['SURVEY_DATE'].isna()


# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab = pd.crosstab(glathida_rgis_edited['is_na_servey_date'], glathida_rgis_edited['GlaThiDa_ID'])

print(crosstab)

In [ ]:
pd.unique(glathida_rgis["SURVEY_DATE"])


In [ ]:
columns_of_interest = ['GlaThiDa_ID', 'SURVEY_DATE']

# Filtrer rækkerne, hvor værdien i 'column1' er 501
filtered_data = glathida_rgis[glathida_rgis['GlaThiDa_ID'] == 500]

# Vælg de to kolonner fra de filtrerede data
subset_data = filtered_data[columns_of_interest]

# Udskriv de første par rækker af de valgte kolonner for at kontrollere resultatet
print(subset_data.head())


In [ ]:
columns_of_interest = ['GlaThiDa_ID', 'SURVEY_DATE']

# Filtrer rækkerne, hvor værdien i 'column1' er 501
filtered_data = glathida_rgis[glathida_rgis['GlaThiDa_ID'] == 502]

# Vælg de to kolonner fra de filtrerede data
subset_data = filtered_data[columns_of_interest]

# Udskriv de første par rækker af de valgte kolonner for at kontrollere resultatet
print(subset_data.head())


In [ ]:
# Lag en krysstabell mellom 'is_na_floor' og 'policy_ID'
crosstab = pd.crosstab(subset['is_na_remarks'], subset['is_na_data_flag'])

print(crosstab)

# Final glathida_rgis - dataset

In [ ]:
path_O1_shp = '/Users/emma/OGGM/rgi/RGIV62/00_rgi62_regions/00_rgi62_O1Regions.shp' # shp file of regional boundaries

# Import 20-bin gridded training dataset
metadata_file = "//Users/emma/aml/GroupProject/metadata19-001.csv"
glathida_rgis = pd.read_csv(metadata_file, low_memory=False)

In [ ]:
# Fjern rækkerne hvor 'DATA_FLAG' er lig med 1
glathida_rgis_edited = glathida_rgis[glathida_rgis['DATA_FLAG'] != 1]

# Fjern kolonnen "DATA_FLAG"
glathida_rgis_edited.drop("DATA_FLAG", axis=1, inplace=True)

In [ ]:
glathida_rgis_edited = glathida_rgis_edited.drop(['GLACIER_NAME', 'PROFILE_ID', 'SURVEY_DATE', 'ELEVATION'], axis=1)

In [ ]:
glathida_rgis_edited = glathida_rgis_edited.drop(['THICKNESS_UNCERTAINTY'], axis=1)

In [ ]:
# Opret et subset, hvor 'THICKNESS' er forskellig fra 0
glathida_rgis_edited = glathida_rgis_edited[glathida_rgis_edited['THICKNESS'] != 0]

In [ ]:
# Opret et subset, hvor rækker med NaN-værdier i kolonnen 'slope' fjernes
glathida_rgis_edited = glathida_rgis_edited.dropna(subset=['Slope'])
glathida_rgis_edited = glathida_rgis_edited.dropna(subset=['vx'])
glathida_rgis_edited = glathida_rgis_edited.dropna(subset=['dvx_dx'])
glathida_rgis_edited = glathida_rgis_edited.dropna(subset=['dvx_dy'])
glathida_rgis_edited = glathida_rgis_edited.dropna(subset=['smb'])

In [ ]:
# Udskriv de første par rækker for at kontrollere resultatet
print(glathida_rgis_edited.head())

# Alternativt kan du tjekke om der er NaN værdier i slope kolonnen i det nye subset
print(f"Antal NaN-værdier i 'slope' efter fjernelse: {glathida_rgis_edited['Slope'].isna().sum()}")


In [ ]:
pd.unique(glathida_rgis['REMARKS'])

In [ ]:
pd.unique(glathida_rgis_edited['REMARKS'])

In [ ]:
len(pd.unique(glathida_rgis['RGIId']))

In [ ]:
#Udskifter alle nan i REMARKS til none/unknown"
glathida_rgis_edited['REMARKS'].fillna('none/unknown', inplace=True)

In [ ]:
import re

# Antag at "glathida_rgis.csv" er dit datasæt
# Læs datasættet ind i en pandas DataFrame

def count_na_and_nan(dataframe):
    counts = {}
    for column in dataframe.columns:
        # Konverter kolonneværdier til strenge, hvis de ikke allerede er strenge
        column_values_as_strings = dataframe[column].astype(str)
        
        # Saml alle værdier fra kolonnen til én lang streng
        column_text = " ".join(column_values_as_strings)
        
        # Find forekomster af 'na' og 'nan' i kolonnen
        na_matches = re.findall(r'\bna\b', column_text)
        nan_matches = re.findall(r'\bnan\b', column_text)
        
        # Tæl forekomsterne
        na_count = len(na_matches)
        nan_count = len(nan_matches)
        
        # Gem tællingerne for denne kolonne i dictionaryen
        counts[column] = {'na_count': na_count, 'nan_count': nan_count}
        
    return counts

# Kald funktionen og gem resultatet
counts = count_na_and_nan(glathida_rgis_edited)

# Udskriv resultaterne
for column, count_dict in counts.items():
    print(f"Column '{column}':")
    print(f"Occurrences of 'na': {count_dict['na_count']}")
    print(f"Occurrences of 'nan': {count_dict['nan_count']}")
    print()

In [ ]:
# Beregn 'slope_total' for hver række
glathida_rgis_edited['slope_total'] = np.sqrt(glathida_rgis_edited['slope_lat']**2 + glathida_rgis_edited['slope_lon']**2)

glathida_rgis_edited['slope_total_gfa'] = np.sqrt(glathida_rgis_edited['slope_lat_gfa']**2 + glathida_rgis_edited['slope_lon_gfa']**2)

# Gem det opdaterede datasæt tilbage til en ny CSV-fil
#df.to_csv('glathida_rgis_edited_with_slope_total.csv', index=False)

#print("Beregningen af 'slope_total' er fuldført, og resultatet er gemt i 'glathida_rgis_edited_with_slope_total.csv'")

In [ ]:
glathida_rgis_edited = glathida_rgis_edited.drop(['slope_lat_gf50', 'slope_lon_gf50', 'slope_lat_gf100', 'slope_lon_gf100',
                                                  'slope_lat_gf150', 'slope_lon_gf150','slope_lat_gf300', 'slope_lon_gf300',
                                                 'slope_lat_gf450', 'slope_lon_gf450',
                                                 'curv_50', 'curv_300',
                                                 'aspect_50', 'aspect_300',
                                                 'vx_gf50', 'vx_gf100', 'vx_gf150', 'vx_gf300',  'vx_gf450',
                                                 'vy_gf50', 'vy_gf100', 'vy_gf150', 'vy_gf300', 'vy_gf450',
                                                 ], axis=1)

In [ ]:
# Opret et subset af de rækker, hvor 'RGIId' mangler værdier
subset_missing_RGIId = glathida_rgis_edited[glathida_rgis_edited['vx'].isna()]

# Se de første rækker i subsettet
print(subset_missing_RGIId)


# Impact encoding

In [ ]:
for col in glathida_rgis_edited.columns:
    print(f"{col}: {glathida_rgis_edited[col].dtype}")

In [ ]:
# Beregn gennemsnitlig huspris for hver by
impact_encoding1 = glathida_rgis_edited.groupby('POINT_ID')['THICKNESS'].mean().to_dict()

# Erstat byerne med de beregnede gennemsnitsværdier
glathida_rgis_edited['PPOINT_ID_encoded'] = glathida_rgis_edited['POINT_ID'].map(impact_encoding1)


# Beregn gennemsnitlig huspris for hver by
impact_encoding2 = glathida_rgis_edited.groupby('POLITICAL_UNIT')['THICKNESS'].mean().to_dict()

# Erstat byerne med de beregnede gennemsnitsværdier
glathida_rgis_edited['POLITICAL_UNIT_encoded'] = glathida_rgis_edited['POLITICAL_UNIT'].map(impact_encoding2)



# Beregn gennemsnitlig huspris for hver by
impact_encoding3 = glathida_rgis_edited.groupby('REMARKS')['THICKNESS'].mean().to_dict()

# Erstat byerne med de beregnede gennemsnitsværdier
glathida_rgis_edited['REMARKS_encoded'] = glathida_rgis_edited['REMARKS'].map(impact_encoding3)



print(glathida_rgis_edited)

In [ ]:
# Antag at din data er gemt i en DataFrame kaldet 'data' med kolonnenavne 'feature_1', 'feature_2', ..., 'feature_n' og 'target_variable'

# Beregn korrelationskoefficienter mellem alle funktioner og målvariabelen
correlations = glathida_rgis.corr()['THICKNESS'].abs().sort_values(ascending=False)

# Vælg de 30 mest korrelerede funktioner (ekskluderer målvariabelen selv)
top_correlated_features = correlations.drop('THICKNESS').head(30)

# Udskriv de 30 mest korrelerede funktioner og deres korrelationskoefficienter
for feature, correlation in top_correlated_features.items():
    print(f"{feature}: {correlation}")

# Merging data

## RGI10

In [ ]:
path_O1_shp = '/Users/emma/OGGM/rgi/RGIV62/00_rgi62_regions/00_rgi62_O1Regions.shp' # shp file of regional boundaries

# Import 20-bin gridded training dataset
glacier_data_RGI10 = "//Users/emma/aml/GroupProject/glacier_data_RGI10.csv"
glacier_data_RGI10 = pd.read_csv(glacier_data_RGI10, low_memory=False)

In [ ]:
glacier_data_RGI10

In [ ]:
# Opret en liste med de ønskede værdier
desired_values10 = [f'RGI60-10.{i:05d}' for i in range(1, 5152)]

# Filtrer datasættet
subset_df10 = glathida_rgis_edited[glathida_rgis_edited['RGIId'].isin(desired_values10)]

In [ ]:
subset_df10

In [ ]:
glacier_data_RGI10 = glacier_data_RGI10.rename(columns={'glacierID': 'RGIId'})

In [ ]:
common_values10 = glacier_data_RGI10['RGIId'].unique()
filtered_df10 = glathida_rgis_edited[glathida_rgis_edited['RGIId'].isin(common_values10)]

In [ ]:
#filtered_df10_latent = latent_space_data[latent_space_data['RGIId'].isin(common_values10)]

In [ ]:
merged_df10 = pd.merge(glacier_data_RGI10, filtered_df10, on='RGIId', how='inner')

In [ ]:
merged_df10

## RGI 11

In [ ]:
path_O1_shp = '/Users/emma/OGGM/rgi/RGIV62/00_rgi62_regions/00_rgi62_O1Regions.shp' # shp file of regional boundaries

# Import 20-bin gridded training dataset
glacier_data_RGI11 = "//Users/emma/aml/GroupProject/glacier_data_RGI11.csv"
glacier_data_RGI11 = pd.read_csv(glacier_data_RGI11, low_memory=False)

In [ ]:
glacier_data_RGI11

In [ ]:
# Opret en liste med de ønskede værdier
desired_values11 = [f'RGI60-11.{i:05d}' for i in range(1, 3928)]

# Filtrer datasættet
subset_df11 = glathida_rgis_edited[glathida_rgis_edited['RGIId'].isin(desired_values11)]

In [ ]:
subset_df11

In [ ]:
glacier_data_RGI11 = glacier_data_RGI11.rename(columns={'glacierID': 'RGIId'})

In [ ]:
common_values11 = glacier_data_RGI11['RGIId'].unique()
filtered_df11 = glathida_rgis_edited[glathida_rgis_edited['RGIId'].isin(common_values11)]

In [ ]:
merged_df11 = pd.merge(glacier_data_RGI11, filtered_df11, on='RGIId', how='inner')

In [ ]:
merged_df11

## RGI 12

In [ ]:
path_O1_shp = '/Users/emma/OGGM/rgi/RGIV62/00_rgi62_regions/00_rgi62_O1Regions.shp' # shp file of regional boundaries

# Import 20-bin gridded training dataset
glacier_data_RGI12 = "//Users/emma/aml/GroupProject/glacier_data_RGI12.csv"
glacier_data_RGI12 = pd.read_csv(glacier_data_RGI12, low_memory=False)
glacier_data_RGI12

In [ ]:
# Vis datatyperne for hver kolonne
print(glacier_data_RGI12.dtypes)

In [ ]:
# Opret en liste med de ønskede værdier
desired_values12 = [f'RGI60-12.{i:05d}' for i in range(1, 1889)]

# Filtrer datasættet
subset_df12 = glathida_rgis_edited[glathida_rgis_edited['RGIId'].isin(desired_values12)]

In [ ]:
glacier_data_RGI12 = glacier_data_RGI12.rename(columns={'glacierID': 'RGIId'})

In [ ]:
common_values12 = glacier_data_RGI12['RGIId'].unique()
filtered_df12 = glathida_rgis_edited[glathida_rgis_edited['RGIId'].isin(common_values12)]

In [ ]:
merged_df12 = pd.merge(glacier_data_RGI12, filtered_df12, on='RGIId', how='inner')
merged_df12

## RGI 13

In [ ]:
path_O1_shp = '/Users/emma/OGGM/rgi/RGIV62/00_rgi62_regions/00_rgi62_O1Regions.shp' # shp file of regional boundaries

# Import 20-bin gridded training dataset
glacier_data_RGI13 = "//Users/emma/aml/GroupProject/glacier_data_RGI13.csv"
glacier_data_RGI13 = pd.read_csv(glacier_data_RGI13, low_memory=False)
glacier_data_RGI13

In [ ]:
# Vis datatyperne for hver kolonne
print(glacier_data_RGI13.dtypes)

In [ ]:
# Opret en liste med de ønskede værdier
desired_values13 = [f'RGI60-13.{i:05d}' for i in range(1, 54430)]

# Filtrer datasættet
subset_df13 = glathida_rgis_edited[glathida_rgis_edited['RGIId'].isin(desired_values13)]

In [ ]:
glacier_data_RGI13 = glacier_data_RGI13.rename(columns={'glacierID': 'RGIId'})

In [ ]:
common_values13 = glacier_data_RGI13['RGIId'].unique()
filtered_df13 = glathida_rgis_edited[glathida_rgis_edited['RGIId'].isin(common_values13)]

In [ ]:
merged_df13 = pd.merge(glacier_data_RGI13, filtered_df13, on='RGIId', how='inner')
merged_df13

## VAE Latent space data (62 dimensions)

In [ ]:
# Import 20-bin gridded training dataset
train_latent_spaceVAE = "//Users/emma/aml/GroupProject/train_latent_spaceVAE.csv"
train_latent_spaceVAE = pd.read_csv(train_latent_spaceVAE, low_memory=False)
train_latent_spaceVAE

In [ ]:
 train_latent_space.describe()

In [ ]:
# Import 20-bin gridded training dataset
val_latent_spaceVAE = "//Users/emma/aml/GroupProject/val_latent_spaceVAE.csv"
val_latent_spaceVAE = pd.read_csv(val_latent_spaceVAE, low_memory=False)
val_latent_spaceVAE

In [ ]:
# Sammensæt datasættene
latent_space_data = pd.concat([train_latent_spaceVAE, val_latent_spaceVAE], axis=0)

In [ ]:
# Gem det nye datasæt
latent_space_data.to_csv('latent_space_data.csv', index=False)

In [ ]:
latent_space_data

In [ ]:
latent_space_dataVAE = latent_space_dataVAE.rename(columns={'glacier_id': 'RGIId'})

## RGI 16

In [ ]:
path_O1_shp = '/Users/emma/OGGM/rgi/RGIV62/00_rgi62_regions/00_rgi62_O1Regions.shp' # shp file of regional boundaries

# Import 20-bin gridded training dataset
glacier_data_RGI16 = "/Users/emma/aml/AML autoencoder data/glacier_data_RGI16.csv"
glacier_data_RGI16 = pd.read_csv(glacier_data_RGI16, low_memory=False)

In [ ]:
glacier_data_RGI16

In [ ]:
# Opret en liste med de ønskede værdier
desired_values16 = [f'RGI60-16.{i:05d}' for i in range(1, 2940)]

# Filtrer datasættet
subset_df16 = glathida_rgis_edited[glathida_rgis_edited['RGIId'].isin(desired_values16)]

In [ ]:
subset_df16

In [ ]:
glacier_data_RGI16 = glacier_data_RGI16.rename(columns={'glacierID': 'RGIId'})

In [ ]:
common_values16 = glacier_data_RGI16['RGIId'].unique()
filtered_df16 = glathida_rgis_edited[glathida_rgis_edited['RGIId'].isin(common_values16)]

In [ ]:
filtered_df16_latent = latent_space_data[latent_space_data['RGIId'].isin(common_values16)]

In [ ]:
merged_df16 = pd.merge(glacier_data_RGI16, filtered_df16, on='RGIId', how='inner')

In [ ]:
merged_df16

## RGI 17

In [ ]:
path_O1_shp = '/Users/emma/OGGM/rgi/RGIV62/00_rgi62_regions/00_rgi62_O1Regions.shp' # shp file of regional boundaries

# Import 20-bin gridded training dataset
glacier_data_RGI17 = "/Users/emma/aml/AML autoencoder data/glacier_data_RGI17.csv"
glacier_data_RGI17 = pd.read_csv(glacier_data_RGI17, low_memory=False)

In [ ]:
glacier_data_RGI17

In [ ]:
# Opret en liste med de ønskede værdier
desired_values17 = [f'RGI60-17.{i:05d}' for i in range(1, 15909)]

# Filtrer datasættet
subset_df17 = glathida_rgis_edited[glathida_rgis_edited['RGIId'].isin(desired_values17)]

In [ ]:
subset_df17

In [ ]:
glacier_data_RGI17 = glacier_data_RGI17.rename(columns={'glacierID': 'RGIId'})

In [ ]:
common_values17 = glacier_data_RGI17['RGIId'].unique()
filtered_df17 = glathida_rgis_edited[glathida_rgis_edited['RGIId'].isin(common_values17)]

In [ ]:
filtered_df17_latent = latent_space_data[latent_space_data['RGIId'].isin(common_values17)]

In [ ]:
merged_df17 = pd.merge(glacier_data_RGI17, filtered_df17, on='RGIId', how='inner')

In [ ]:
merged_df17

## RGI 18

In [ ]:
path_O1_shp = '/Users/emma/OGGM/rgi/RGIV62/00_rgi62_regions/00_rgi62_O1Regions.shp' # shp file of regional boundaries
 
# Import 20-bin gridded training dataset
glacier_data_RGI18 = "/Users/emma/aml/AML autoencoder data/glacier_data_RGI18.csv"
glacier_data_RGI18 = pd.read_csv(glacier_data_RGI18, low_memory=False)

In [ ]:
glacier_data_RGI18

In [ ]:
# Opret en liste med de ønskede værdier
desired_values18 = [f'RGI60-18.{i:05d}' for i in range(1, 3538)]

# Filtrer datasættet
subset_df18 = glathida_rgis_edited[glathida_rgis_edited['RGIId'].isin(desired_values18)]

In [ ]:
subset_df18

In [ ]:
glacier_data_RGI18 = glacier_data_RGI18.rename(columns={'glacierID': 'RGIId'})

In [ ]:
common_values18 = glacier_data_RGI18['RGIId'].unique()
filtered_df18 = glathida_rgis_edited[glathida_rgis_edited['RGIId'].isin(common_values18)]

In [ ]:
filtered_df18_latent = latent_space_data[latent_space_data['RGIId'].isin(common_values18)]

In [ ]:
merged_df18 = pd.merge(glacier_data_RGI18, filtered_df18, on='RGIId', how='inner')

In [ ]:
merged_df18

## RGI 19

In [ ]:
# Import 20-bin gridded training dataset
glacier_data_RGI19 = "/Users/emma/aml/AML autoencoder data/glacier_data_RGI19.csv"
glacier_data_RGI19 = pd.read_csv(glacier_data_RGI19, low_memory=False)

In [ ]:
glacier_data_RGI19

In [ ]:
# Opret en liste med de ønskede værdier
desired_values19 = [f'RGI60-19.{i:05d}' for i in range(1, 2753)]

# Filtrer datasættet
subset_df19 = glathida_rgis_edited[glathida_rgis_edited['RGIId'].isin(desired_values19)]

In [ ]:
subset_df19

In [ ]:
glacier_data_RGI19 = glacier_data_RGI19.rename(columns={'glacierID': 'RGIId'})

In [ ]:
common_values19 = glacier_data_RGI19['RGIId'].unique()
filtered_df19 = glathida_rgis_edited[glathida_rgis_edited['RGIId'].isin(common_values19)]

In [ ]:
filtered_df19_latent = latent_space_data[latent_space_data['RGIId'].isin(common_values19)]

In [ ]:
merged_df19 = pd.merge(glacier_data_RGI19, filtered_df19, on='RGIId', how='inner')

In [ ]:
merged_df19

# VAE Latent space data 24, 32, 48, 62 dimensions

In [ ]:
# Import 20-bin gridded training dataset
glacier_data_latent_space24 = "/Users/emma/aml/AML autoencoder data/latent_space_24.csv"
glacier_data_latent_space24 = pd.read_csv(glacier_data_latent_space24, low_memory=False)

In [ ]:
glacier_data_latent_space24

In [ ]:
glacier_data_latent_space24 = glacier_data_latent_space24.rename(columns={'glacier_id': 'RGIId'})

In [ ]:
# Import 20-bin gridded training dataset
glacier_data_latent_space32 = "/Users/emma/aml/AML autoencoder data/latent_space_32.csv"
glacier_data_latent_space32 = pd.read_csv(glacier_data_latent_space32, low_memory=False)

In [ ]:
glacier_data_latent_space32 = glacier_data_latent_space32.rename(columns={'glacier_id': 'RGIId'})

In [ ]:
# Import 20-bin gridded training dataset
glacier_data_latent_space48 = "/Users/emma/aml/AML autoencoder data/latent_space_48.csv"
glacier_data_latent_space48 = pd.read_csv(glacier_data_latent_space48, low_memory=False)

In [ ]:
glacier_data_latent_space48 = glacier_data_latent_space48.rename(columns={'glacier_id': 'RGIId'})

In [ ]:
# Import 20-bin gridded training dataset
glacier_data_latent_space64 = "/Users/emma/aml/AML autoencoder data/latent_space_64.csv"
glacier_data_latent_space64 = pd.read_csv(glacier_data_latent_space64, low_memory=False)

In [ ]:
glacier_data_latent_space64 = glacier_data_latent_space64.rename(columns={'glacier_id': 'RGIId'})

In [ ]:
glacier_data_latent_space64

## RGI 10 - Latent space

In [ ]:
filtered_df10_latent24 = glacier_data_latent_space24[glacier_data_latent_space24['RGIId'].isin(common_values10)]

In [ ]:
merged_df10_latent24 = pd.merge(merged_df10, filtered_df10_latent24, on='RGIId', how='inner')

In [ ]:
filtered_df10_latent32 = glacier_data_latent_space32[glacier_data_latent_space32['RGIId'].isin(common_values10)]

In [ ]:
merged_df10_latent32 = pd.merge(merged_df10, filtered_df10_latent32, on='RGIId', how='inner')

In [ ]:
filtered_df10_latent48 = glacier_data_latent_space48[glacier_data_latent_space48['RGIId'].isin(common_values10)]

In [ ]:
merged_df10_latent48 = pd.merge(merged_df10, filtered_df10_latent48, on='RGIId', how='inner')

In [ ]:
filtered_df10_latent64 = glacier_data_latent_space64[glacier_data_latent_space64['RGIId'].isin(common_values10)]

In [ ]:
merged_df10_latent64 = pd.merge(merged_df10, filtered_df10_latent64, on='RGIId', how='inner')

## RGI 11 - Latent space

In [ ]:
filtered_df11_latent24 = glacier_data_latent_space24[glacier_data_latent_space24['RGIId'].isin(common_values11)]

In [ ]:
merged_df11_latent24 = pd.merge(merged_df11, filtered_df11_latent24, on='RGIId', how='inner')

In [ ]:
filtered_df11_latent32 = glacier_data_latent_space32[glacier_data_latent_space32['RGIId'].isin(common_values11)]

In [ ]:
merged_df11_latent32 = pd.merge(merged_df11, filtered_df11_latent32, on='RGIId', how='inner')

In [ ]:
filtered_df11_latent48 = glacier_data_latent_space48[glacier_data_latent_space48['RGIId'].isin(common_values11)]

In [ ]:
merged_df11_latent48 = pd.merge(merged_df11, filtered_df11_latent48, on='RGIId', how='inner')

In [ ]:
filtered_df11_latent64 = glacier_data_latent_space64[glacier_data_latent_space64['RGIId'].isin(common_values11)]

In [ ]:
merged_df11_latent64 = pd.merge(merged_df11, filtered_df11_latent64, on='RGIId', how='inner')

## RGI 12 - Latent space

In [ ]:
filtered_df12_latent24 = glacier_data_latent_space24[glacier_data_latent_space24['RGIId'].isin(common_values12)]

In [ ]:
merged_df12_latent24 = pd.merge(merged_df12, filtered_df12_latent24, on='RGIId', how='inner')

In [ ]:
filtered_df12_latent32 = glacier_data_latent_space32[glacier_data_latent_space32['RGIId'].isin(common_values12)]

In [ ]:
merged_df12_latent32 = pd.merge(merged_df12, filtered_df12_latent32, on='RGIId', how='inner')

In [ ]:
filtered_df12_latent48 = glacier_data_latent_space48[glacier_data_latent_space48['RGIId'].isin(common_values12)]

In [ ]:
merged_df12_latent48 = pd.merge(merged_df12, filtered_df12_latent48, on='RGIId', how='inner')

In [ ]:
filtered_df12_latent64 = glacier_data_latent_space64[glacier_data_latent_space64['RGIId'].isin(common_values12)]

In [ ]:
merged_df12_latent64 = pd.merge(merged_df12, filtered_df12_latent64, on='RGIId', how='inner')

## RGI 13 - Latent space

In [ ]:
filtered_df13_latent24 = glacier_data_latent_space24[glacier_data_latent_space24['RGIId'].isin(common_values13)]

In [ ]:
merged_df13_latent24 = pd.merge(merged_df13, filtered_df13_latent24, on='RGIId', how='inner')

In [ ]:
filtered_df13_latent32 = glacier_data_latent_space32[glacier_data_latent_space32['RGIId'].isin(common_values13)]

In [ ]:
merged_df13_latent32 = pd.merge(merged_df13, filtered_df13_latent32, on='RGIId', how='inner')

In [ ]:
filtered_df13_latent48 = glacier_data_latent_space48[glacier_data_latent_space48['RGIId'].isin(common_values13)]

In [ ]:
merged_df13_latent48 = pd.merge(merged_df13, filtered_df13_latent48, on='RGIId', how='inner')

In [ ]:
filtered_df13_latent64 = glacier_data_latent_space64[glacier_data_latent_space64['RGIId'].isin(common_values13)]

In [ ]:
merged_df13_latent64 = pd.merge(merged_df13, filtered_df13_latent64, on='RGIId', how='inner')

## RGI 16 - Latent space

In [ ]:
#glacier_data_latent_space24

In [ ]:
filtered_df16_latent24 = glacier_data_latent_space24[glacier_data_latent_space24['RGIId'].isin(common_values16)]

In [ ]:
merged_df16_latent24 = pd.merge(merged_df16, filtered_df16_latent24, on='RGIId', how='inner')

In [ ]:
filtered_df16_latent32 = glacier_data_latent_space32[glacier_data_latent_space32['RGIId'].isin(common_values16)]

In [ ]:
merged_df16_latent32 = pd.merge(merged_df16, filtered_df16_latent32, on='RGIId', how='inner')

In [ ]:
filtered_df16_latent48 = glacier_data_latent_space48[glacier_data_latent_space48['RGIId'].isin(common_values16)]

In [ ]:
merged_df16_latent48 = pd.merge(merged_df16, filtered_df16_latent48, on='RGIId', how='inner')

In [ ]:
filtered_df16_latent64 = glacier_data_latent_space64[glacier_data_latent_space64['RGIId'].isin(common_values16)]

In [ ]:
merged_df16_latent64 = pd.merge(merged_df16, filtered_df16_latent64, on='RGIId', how='inner')

## RGI 17 - Latent space

In [ ]:
merged_df17

In [ ]:
filtered_df17_latent24 = glacier_data_latent_space24[glacier_data_latent_space24['RGIId'].isin(common_values17)]

In [ ]:
merged_df17_latent24 = pd.merge(merged_df17, filtered_df17_latent24, on='RGIId', how='inner')

In [ ]:
filtered_df17_latent32 = glacier_data_latent_space32[glacier_data_latent_space32['RGIId'].isin(common_values17)]

In [ ]:
merged_df17_latent32 = pd.merge(merged_df17, filtered_df17_latent32, on='RGIId', how='inner')

In [ ]:
filtered_df17_latent48 = glacier_data_latent_space48[glacier_data_latent_space48['RGIId'].isin(common_values17)]

In [ ]:
merged_df17_latent48 = pd.merge(merged_df17, filtered_df17_latent48, on='RGIId', how='inner')

In [ ]:
filtered_df17_latent64 = glacier_data_latent_space64[glacier_data_latent_space64['RGIId'].isin(common_values17)]

In [ ]:
merged_df17_latent64 = pd.merge(merged_df17, filtered_df17_latent64, on='RGIId', how='inner')

## RGI 18 - Latent space

In [ ]:
filtered_df18_latent24 = glacier_data_latent_space24[glacier_data_latent_space24['RGIId'].isin(common_values18)]

In [ ]:
merged_df18_latent24 = pd.merge(merged_df18, filtered_df18_latent24, on='RGIId', how='inner')

In [ ]:
filtered_df18_latent32 = glacier_data_latent_space32[glacier_data_latent_space32['RGIId'].isin(common_values18)]

In [ ]:
merged_df18_latent32 = pd.merge(merged_df18, filtered_df18_latent32, on='RGIId', how='inner')

In [ ]:
filtered_df18_latent48 = glacier_data_latent_space48[glacier_data_latent_space48['RGIId'].isin(common_values18)]

In [ ]:
merged_df18_latent48 = pd.merge(merged_df18, filtered_df18_latent48, on='RGIId', how='inner')

In [ ]:
filtered_df18_latent64 = glacier_data_latent_space64[glacier_data_latent_space64['RGIId'].isin(common_values18)]

In [ ]:
merged_df18_latent64 = pd.merge(merged_df18, filtered_df18_latent64, on='RGIId', how='inner')

## RGI 19 - Latent space

In [ ]:
filtered_df19_latent24 = glacier_data_latent_space24[glacier_data_latent_space24['RGIId'].isin(common_values19)]

In [ ]:
merged_df19_latent24 = pd.merge(merged_df19, filtered_df19_latent24, on='RGIId', how='inner')

In [ ]:
filtered_df19_latent32 = glacier_data_latent_space32[glacier_data_latent_space32['RGIId'].isin(common_values19)]

In [ ]:
merged_df19_latent32 = pd.merge(merged_df19, filtered_df19_latent32, on='RGIId', how='inner')

In [ ]:
filtered_df19_latent48 = glacier_data_latent_space48[glacier_data_latent_space48['RGIId'].isin(common_values19)]

In [ ]:
merged_df19_latent48 = pd.merge(merged_df19, filtered_df19_latent48, on='RGIId', how='inner')

In [ ]:
filtered_df19_latent64 = glacier_data_latent_space64[glacier_data_latent_space64['RGIId'].isin(common_values19)]

In [ ]:
merged_df19_latent64 = pd.merge(merged_df19, filtered_df19_latent64, on='RGIId', how='inner')

# Final latent space datasets

In [ ]:
latent_space_data_merged24_1 = pd.concat([merged_df10_latent24, merged_df11_latent24], axis=0)

In [ ]:
latent_space_data_merged24_2 = pd.concat([merged_df12_latent24, merged_df13_latent24], axis=0)

In [ ]:
latent_space_data_merged24_3 = pd.concat([merged_df16_latent24, merged_df17_latent24], axis=0)

In [ ]:
latent_space_data_merged24_4 = pd.concat([merged_df18_latent24, merged_df19_latent24], axis=0)

In [ ]:
latent_space_data_merged24_5 = pd.concat([latent_space_data_merged24_1, latent_space_data_merged24_2], axis=0)

In [ ]:
latent_space_data_merged24_6 = pd.concat([latent_space_data_merged24_3, latent_space_data_merged24_4], axis=0)

In [ ]:
latent_space_data_merged24 = pd.concat([latent_space_data_merged24_5, latent_space_data_merged24_6], axis=0)

In [ ]:
latent_space_data_merged24

In [ ]:
latent_space_data_merged24.to_csv('latent_space_data_merged24.csv', index=False)

In [ ]:
latent_space_data_merged32_1 = pd.concat([merged_df10_latent32, merged_df11_latent32], axis=0)

In [ ]:
latent_space_data_merged32_2 = pd.concat([merged_df12_latent32, merged_df13_latent32], axis=0)

In [ ]:
latent_space_data_merged32_3 = pd.concat([merged_df16_latent32, merged_df17_latent32], axis=0)

In [ ]:
latent_space_data_merged32_4 = pd.concat([merged_df18_latent32, merged_df19_latent32], axis=0)

In [ ]:
latent_space_data_merged32_5 = pd.concat([latent_space_data_merged32_1, latent_space_data_merged32_2], axis=0)

In [ ]:
latent_space_data_merged32_6 = pd.concat([latent_space_data_merged32_3, latent_space_data_merged32_4], axis=0)

In [ ]:
latent_space_data_merged32 = pd.concat([latent_space_data_merged32_5, latent_space_data_merged32_6], axis=0)

In [ ]:
latent_space_data_merged32

In [ ]:
latent_space_data_merged32.to_csv('latent_space_data_merged32.csv', index=False)

In [ ]:
latent_space_data_merged48_1 = pd.concat([merged_df10_latent48, merged_df11_latent48], axis=0)
latent_space_data_merged48_2 = pd.concat([merged_df12_latent48, merged_df13_latent48], axis=0)
latent_space_data_merged48_3 = pd.concat([merged_df16_latent48, merged_df17_latent48], axis=0)
latent_space_data_merged48_4 = pd.concat([merged_df18_latent48, merged_df19_latent48], axis=0)
latent_space_data_merged48_5 = pd.concat([latent_space_data_merged48_1, latent_space_data_merged48_2], axis=0)
latent_space_data_merged48_6 = pd.concat([latent_space_data_merged48_3, latent_space_data_merged48_4], axis=0)
latent_space_data_merged48 = pd.concat([latent_space_data_merged48_5, latent_space_data_merged48_6], axis=0)
latent_space_data_merged48

In [ ]:
latent_space_data_merged48.to_csv('latent_space_data_merged48.csv', index=False)

In [ ]:
latent_space_data_merged64_1 = pd.concat([merged_df10_latent64, merged_df11_latent64], axis=0)
latent_space_data_merged64_2 = pd.concat([merged_df12_latent64, merged_df13_latent64], axis=0)
latent_space_data_merged64_3 = pd.concat([merged_df16_latent64, merged_df17_latent64], axis=0)
latent_space_data_merged64_4 = pd.concat([merged_df18_latent64, merged_df19_latent64], axis=0)
latent_space_data_merged64_5 = pd.concat([latent_space_data_merged64_1, latent_space_data_merged64_2], axis=0)
latent_space_data_merged64_6 = pd.concat([latent_space_data_merged64_3, latent_space_data_merged64_4], axis=0)
latent_space_data_merged64 = pd.concat([latent_space_data_merged64_5, latent_space_data_merged64_6], axis=0)
latent_space_data_merged64

In [ ]:
latent_space_data_merged64.to_csv('latent_space_data_merged64.csv', index=False)

In [ ]:

# Beregn gennemsnittet indenfor hver RGIId
mean_values = latent_space_data_merged24.groupby('RGIId')[['POINT_LAT','POINT_LON']].transform('mean')

# Opret de lokale koordinater ved at subtrahere gennemsnittet
latent_space_data_merged24['local_longitude'] = latent_space_data_merged24['POINT_LON'] - mean_values['POINT_LON']
latent_space_data_merged24['local_latitude'] = latent_space_data_merged24['POINT_LAT'] - mean_values['POINT_LAT']

In [ ]:
# Beregn gennemsnittet indenfor hver RGIId
mean_values = latent_space_data_merged32.groupby('RGIId')[['POINT_LAT','POINT_LON']].transform('mean')

# Opret de lokale koordinater ved at subtrahere gennemsnittet
latent_space_data_merged32['local_longitude'] = latent_space_data_merged32['POINT_LON'] - mean_values['POINT_LON']
latent_space_data_merged32['local_latitude'] = latent_space_data_merged32['POINT_LAT'] - mean_values['POINT_LAT']

In [ ]:
# Beregn gennemsnittet indenfor hver RGIId
mean_values = latent_space_data_merged48.groupby('RGIId')[['POINT_LAT','POINT_LON']].transform('mean')

# Opret de lokale koordinater ved at subtrahere gennemsnittet
latent_space_data_merged48['local_longitude'] = latent_space_data_merged48['POINT_LON'] - mean_values['POINT_LON']
latent_space_data_merged48['local_latitude'] = latent_space_data_merged48['POINT_LAT'] - mean_values['POINT_LAT']

In [ ]:
# Beregn gennemsnittet indenfor hver RGIId
mean_values = latent_space_data_merged64.groupby('RGIId')[['POINT_LAT','POINT_LON']].transform('mean')

# Opret de lokale koordinater ved at subtrahere gennemsnittet
latent_space_data_merged64['local_longitude'] = latent_space_data_merged64['POINT_LON'] - mean_values['POINT_LON']
latent_space_data_merged64['local_latitude'] = latent_space_data_merged64['POINT_LAT'] - mean_values['POINT_LAT']

In [ ]:
latent_space_data_merged64

# AE 64

In [ ]:
# Import 20-bin gridded training dataset
glacier_data_latent_space64AE = "/Users/emma/aml/AML autoencoder data/latent_space_64_AE.csv"
glacier_data_latent_space64AE = pd.read_csv(glacier_data_latent_space64AE, low_memory=False)

In [ ]:
glacier_data_latent_space64AE = glacier_data_latent_space64AE.rename(columns={'glacier_id': 'RGIId'})

In [ ]:
glacier_data_latent_space64_AE = pd.merge(glacier_data_latent_space64AE, glathida_rgis_edited, on='RGIId', how='inner')

In [ ]:
glacier_data_latent_space64_AE

In [ ]:
# Beregn gennemsnittet indenfor hver RGIId
mean_values = glacier_data_latent_space64_AE.groupby('RGIId')[['POINT_LAT','POINT_LON']].transform('mean')

# Opret de lokale koordinater ved at subtrahere gennemsnittet
glacier_data_latent_space64_AE['local_longitude'] = glacier_data_latent_space64_AE['POINT_LON'] - mean_values['POINT_LON']
glacier_data_latent_space64_AE['local_latitude'] = glacier_data_latent_space64_AE['POINT_LAT'] - mean_values['POINT_LAT']

In [ ]:
glacier_data_latent_space64_AE = glacier_data_latent_space64_AE.drop(['RGIId', 'GlaThiDa_ID', 'POINT_ID', 'POINT_LAT', 'POINT_LON', 'Slope', 
                                                              'PPOINT_ID_encoded', 'POLITICAL_UNIT_encoded', 'slope_lat_gfa', 'slope_lon_gfa',
                                                                     'curv_gfa', 'aspect_gfa', 'vx_gfa', 'vy_gfa', 'slope_total_gfa'], axis=1)

In [ ]:
# Udtræk de numeriske kolonner
numeric_columns_latent_space64_AE = glacier_data_latent_space64_AE.select_dtypes(include=['int', 'float']).columns

# Opret et subset af de numeriske variable
numeric_columns_latent_space64_AE = glacier_data_latent_space64_AE[numeric_columns_latent_space64_AE]

print("Subset af numeriske variable for 64 AE:")
print(numeric_columns_latent_space64_AE)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

X = numeric_columns_latent_space64_AE.drop(['THICKNESS', 'ith_f', 'ith_m'], axis=1)
y = numeric_columns_latent_space64_AE['THICKNESS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Opret en Random Forest Regressor model
rf_regressor = RandomForestRegressor(n_estimators=10, random_state=42)

# Træn modellen på træningsdataene
rf_regressor.fit(X_train, y_train)

# Lav forudsigelser på testdataene
predictions = rf_regressor.predict(X_test)

# Evaluer præstationen med Mean Absolute Error (MAE)
mae = mean_absolute_error(y_test, predictions)
print("Mean Absolute Error (MAE):", mae)

# Fitting models on VAE laten space data

## For 24

In [ ]:
# Iterere gennem kolonnerne og print datatyperne
for col in latent_space_data_merged24.columns:
    print(f'{col}: {latent_space_data_merged24[col].dtype}')

In [ ]:
latent_space_data_merged24_subset = latent_space_data_merged24.drop(['RGIId', 'GlaThiDa_ID', 'POINT_ID', 'POINT_LAT', 'POINT_LON', 'Slope', 
                                                              'PPOINT_ID_encoded', 'POLITICAL_UNIT_encoded', 'slope_lat_gfa', 'slope_lon_gfa',
                                                                     'curv_gfa', 'aspect_gfa', 'vx_gfa', 'vy_gfa', 'slope_total_gfa'], axis=1)

In [ ]:
# Udtræk de numeriske kolonner
numeric_columns_latent_space24 = latent_space_data_merged24_subset.select_dtypes(include=['int', 'float']).columns

# Opret et subset af de numeriske variable
numeric_columns_latent_space24 = latent_space_data_merged24_subset[numeric_columns_latent_space24]

print("Subset af numeriske variable for 24:")
print(numeric_columns_latent_space24)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

X = numeric_columns_latent_space24.drop('THICKNESS', axis=1)
y = numeric_columns_latent_space24['THICKNESS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Opret en Random Forest Regressor model
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)

# Træn modellen på træningsdataene
rf_regressor.fit(X_train, y_train)

# Lav forudsigelser på testdataene
predictions = rf_regressor.predict(X_test)

# Evaluer præstationen med Mean Absolute Error (MAE)
mae = mean_absolute_error(y_test, predictions)
print("Mean Absolute Error (MAE):", mae)

## for 32

In [ ]:
latent_space_data_merged32_subset = latent_space_data_merged32.drop(['RGIId', 'GlaThiDa_ID', 'POINT_ID', 'POINT_LAT', 'POINT_LON', 'Slope', 
                                                              'PPOINT_ID_encoded', 'POLITICAL_UNIT_encoded', 'slope_lat_gfa', 'slope_lon_gfa',
                                                                     'curv_gfa', 'aspect_gfa', 'vx_gfa', 'vy_gfa', 'slope_total_gfa'], axis=1)

In [ ]:
# Udtræk de numeriske kolonner
numeric_columns_latent_space32 = latent_space_data_merged32_subset.select_dtypes(include=['int', 'float']).columns

# Opret et subset af de numeriske variable
numeric_columns_latent_space32 = latent_space_data_merged32_subset[numeric_columns_latent_space32]

print("Subset af numeriske variable for 32:")
print(numeric_columns_latent_space32)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

X = numeric_columns_latent_space32.drop(['THICKNESS', 'ith_f', 'ith_m'], axis=1)
y = numeric_columns_latent_space32['THICKNESS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Opret en Random Forest Regressor model
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)

# Træn modellen på træningsdataene
rf_regressor.fit(X_train, y_train)

# Lav forudsigelser på testdataene
predictions = rf_regressor.predict(X_test)

# Evaluer præstationen med Mean Absolute Error (MAE)
mae = mean_absolute_error(y_test, predictions)
print("Mean Absolute Error (MAE):", mae)

## model kun med predictions for tidligere modeller

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

X = glathida_rgis_edited[['ith_m']]
y = glathida_rgis_edited['THICKNESS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Opret en Random Forest Regressor model
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)

# Træn modellen på træningsdataene
rf_regressor.fit(X_train, y_train)

# Lav forudsigelser på testdataene
predictions = rf_regressor.predict(X_test)

# Evaluer præstationen med Mean Absolute Error (MAE)
mae = mean_absolute_error(y_test, predictions)
print("Mean Absolute Error (MAE):", mae)

In [ ]:
#results_ith_m = pd.DataFrame({
#    'Actual': y_test,
#    'Predicted': predictions
#})
print(results_ith_m)

In [ ]:
results_ith_m.to_csv('predictions_vs_actual_ithf.csv', index=False)

In [ ]:
X = glathida_rgis_edited[['ith_f']]
y = glathida_rgis_edited['THICKNESS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Opret en Random Forest Regressor model
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)

# Træn modellen på træningsdataene
rf_regressor.fit(X_train, y_train)

# Lav forudsigelser på testdataene
predictions = rf_regressor.predict(X_test)

# Evaluer præstationen med Mean Absolute Error (MAE)
mae = mean_absolute_error(y_test, predictions)
print("Mean Absolute Error (MAE):", mae)

In [ ]:
results_ith_f = pd.DataFrame({
    'Actual': y_test,
    'Predicted': predictions
})
print(results_ith_f)

In [ ]:
results_ith_f.to_csv('predictions_vs_actual_ithf.csv', index=False)

In [ ]:
df_cleanedm = glathida_rgis_edited.dropna(subset=['ith_m'])

In [ ]:
X = df_cleanedm[['ith_m']]
y = df_cleanedm['THICKNESS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
mae_ithm = mean_absolute_error(y, df_cleanedm.loc[X.index, "ith_m"])

In [ ]:
mae_ithm

In [ ]:
df_cleanedf = glathida_rgis_edited.dropna(subset=['ith_f'])
X = df_cleanedf[['ith_f']]
y = df_cleanedf['THICKNESS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
mae_ithf = mean_absolute_error(y, df_cleanedf.loc[X.index, "ith_f"])

In [ ]:
mae_ithf

## For 48

In [ ]:
latent_space_data_merged48_subset = latent_space_data_merged48.drop(['RGIId', 'GlaThiDa_ID', 'POINT_ID', 'POINT_LAT', 'POINT_LON', 'Slope', 
                                                              'PPOINT_ID_encoded', 'POLITICAL_UNIT_encoded', 'slope_lat_gfa', 'slope_lon_gfa',
                                                                     'curv_gfa', 'aspect_gfa', 'vx_gfa', 'vy_gfa', 'slope_total_gfa'], axis=1)

In [ ]:
# Udtræk de numeriske kolonner
numeric_columns_latent_space48 = latent_space_data_merged48_subset.select_dtypes(include=['int', 'float']).columns

# Opret et subset af de numeriske variable
numeric_columns_latent_space48 = latent_space_data_merged48_subset[numeric_columns_latent_space48]

print("Subset af numeriske variable for :")
print(numeric_columns_latent_space48)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

X = numeric_columns_latent_space32.drop('THICKNESS', axis=1)
y = numeric_columns_latent_space32['THICKNESS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Opret en Random Forest Regressor model
rf_regressor = RandomForestRegressor(n_estimators=10, random_state=42)

# Træn modellen på træningsdataene
rf_regressor.fit(X_train, y_train)

# Lav forudsigelser på testdataene
predictions = rf_regressor.predict(X_test)

# Evaluer præstationen med Mean Absolute Error (MAE)
mae = mean_absolute_error(y_test, predictions)
print("Mean Absolute Error (MAE):", mae)

## For 64

In [ ]:
latent_space_data_merged64_subset = latent_space_data_merged64.drop(['RGIId', 'GlaThiDa_ID', 'POINT_ID', 'POINT_LAT', 'POINT_LON', 'Slope', 
                                                              'PPOINT_ID_encoded', 'POLITICAL_UNIT_encoded', 'slope_lat_gfa', 'slope_lon_gfa',
                                                                     'curv_gfa', 'aspect_gfa', 'vx_gfa', 'vy_gfa', 'slope_total_gfa'], axis=1)

In [ ]:
# Udtræk de numeriske kolonner
numeric_columns_latent_space64 = latent_space_data_merged64_subset.select_dtypes(include=['int', 'float']).columns

# Opret et subset af de numeriske variable
numeric_columns_latent_space64 = latent_space_data_merged64_subset[numeric_columns_latent_space64]


In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

X = numeric_columns_latent_space64.drop('THICKNESS', axis=1)
y = numeric_columns_latent_space64['THICKNESS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Opret en Random Forest Regressor model
rf_regressor = RandomForestRegressor(n_estimators=10, random_state=42)

# Træn modellen på træningsdataene
rf_regressor.fit(X_train, y_train)

# Lav forudsigelser på testdataene
predictions = rf_regressor.predict(X_test)

# Evaluer præstationen med Mean Absolute Error (MAE)
mae = mean_absolute_error(y_test, predictions)
print("Mean Absolute Error (MAE):", mae)
#2.3436336056820033 for 100 træer

# Featureless model

In [ ]:
X = numeric_columns_latent_space64.drop('THICKNESS', axis=1)
y = numeric_columns_latent_space64['THICKNESS']

In [ ]:
np.mean(y)

In [ ]:
np.mean(abs(y-np.mean(y)))

# Comparison

In [ ]:
glacier_data_latent_space64_AE

In [ ]:
glacier_data_latent_space64_AE_2 = pd.merge(glacier_data_latent_space64_AE, sub_df, on='RGIId', how='inner')

In [ ]:
glacier_data_latent_space64_AE = glacier_data_latent_space64_AE.drop(['RGIId', 'GlaThiDa_ID', 'POINT_ID', 'POINT_LAT', 'POINT_LON', 'Slope', 
                                                              'PPOINT_ID_encoded', 'POLITICAL_UNIT_encoded', 'slope_lat_gfa', 'slope_lon_gfa',
                                                                     'curv_gfa', 'aspect_gfa', 'vx_gfa', 'vy_gfa', 'slope_total_gfa'], axis=1)

In [ ]:

# Fjern rækker med NaN-værdier
glacier_data_latent_space64_AE = glacier_data_latent_space64_AE.dropna()


In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# Forudsæt at feature-kolonnerne hedder alt andet end target og forudsigelser fra andre modeller
#features = df.columns.difference([target, 'model1_prediction', 'model2_prediction'])

# Del datasættet i features (X) og target (y)
#X = df[features]
#y = df[target]

# Del datasættet i trænings- og testdata
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialiser og træn XGBoost Regressor
#xgb_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)
#xgb_model.fit(X_train, y_train)

# Lav forudsigelser med den trænede model
#y_pred = xgb_model.predict(X_test)

# Evaluér modelens forudsigelser
mse = mean_squared_error(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"Model Performance:\nMSE: {mse}\nMAE: {mae}\nR²: {r2}")

# Sammenlign med forudsigelser fra andre modeller
model1_mse = mean_squared_error(y_test, glacier_data_latent_space64_AE.loc[X_test.index, 'ith_f'])
model1_mae = mean_absolute_error(y_test, glacier_data_latent_space64_AE.loc[X_test.index, 'ith_f'])
model1_r2 = r2_score(y_test, glacier_data_latent_space64_AE.loc[X_test.index, 'ith_f'])

model2_mse = mean_squared_error(y_test, glacier_data_latent_space64_AE.loc[X_test.index, 'ith_m'])
model2_mae = mean_absolute_error(y_test, glacier_data_latent_space64_AE.loc[X_test.index, 'ith_m'])
model2_r2 = r2_score(y_test, glacier_data_latent_space64_AE.loc[X_test.index, 'ith_m'])

print(f"\nModel 1 (ith_f) Performance:\nMSE: {model1_mse}\nMAE: {model1_mae}\nR²: {model1_r2}")
print(f"\nModel 2 Performance:\nMSE: {model2_mse}\nMAE: {model2_mae}\nR²: {model2_r2}")

# Sammenlign resultaterne
print("\nComparison:")
print(f"Your Model vs Model 1 (ith_f) - MSE Difference: {mse - model1_mse}")
print(f"Your Model vs Model 2 (ith_m) - MSE Difference: {mse - model2_mse}")
print(f"Your Model vs Model 1 (ith_f) - MAE Difference: {mae - model1_mae}")
print(f"Your Model vs Model 2 (ith_m) - MAE Difference: {mae - model2_mae}")
print(f"Your Model vs Model 1 (ith_f) - R² Difference: {r2 - model1_r2}")
print(f"Your Model vs Model 2 (ith_m) - R² Difference: {r2 - model2_r2}")


In [ ]:
df_cleanedf_subset

In [ ]:
# Angiv de to kolonner, du vil beholde
subset_columns = ['THICKNESS', 'ith_f']  # Udskift med dine kolonnenavne

# Opret et subset af DataFrame med kun de to kolonner
df_cleanedf_subset = df_cleanedf[subset_columns]



In [ ]:
results_ith_f2 = pd.DataFrame({
    'Actual': y_test,
    'Predicted': df_cleanedf_subset.loc[X_test.index, 'ith_f']
})
print(results_ith_f2)

In [ ]:
results_ith_f2.to_csv('predictions_vs_actual_ithf2.csv', index=False)

In [ ]:
X

In [ ]:
glathida_rgis_edited = glathida_rgis_edited.dropna(subset=['ith_m'])
glathida_rgis_edited = glathida_rgis_edited.dropna(subset=['ith_f'])

In [ ]:
# Udtræk de numeriske kolonner
numeric_columns = glathida_rgis_edited.select_dtypes(include=['int', 'float']).columns

# Opret et subset af de numeriske variable
numeric_columns = glathida_rgis_edited[numeric_columns]

print("Subset af numeriske variable for:")
print(numeric_columns)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

X = numeric_columns.drop(['ith_m', 'ith_f', 'THICKNESS'],axis=1)
y = numeric_columns['THICKNESS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Opret en Random Forest Regressor model
rf_regressor = RandomForestRegressor(n_estimators=10, random_state=42)

# Træn modellen på træningsdataene
rf_regressor.fit(X_train, y_train)

# Lav forudsigelser på testdataene
predictions = rf_regressor.predict(X_test)

# Evaluer præstationen med Mean Absolute Error (MAE)
mae = mean_absolute_error(y_test, predictions)
print("Mean Absolute Error (MAE):", mae)

In [ ]:
results_metadata = pd.DataFrame({
    'Actual': y_test,
    'Predicted': predictions
})
print(results_metadata)

In [ ]:
results_metadata.to_csv('predictions_vs_actual_metadata.csv', index=False)

In [ ]:
X = df_cleanedf_subset['ith_f']
y = df_cleanedf_subset['THICKNESS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
glathida_rgis_edited.dtypes

In [ ]:
glathida_rgis_edited1 = glathida_rgis_edited.drop(['GlaThiDa_ID', 'POLITICAL_UNIT', 'POINT_ID', 'REMARKS', 'RGIId', 'ith_m', 'ith_f',
                                                  'slope_lat', 'slope_lon', 'slope_lat_gfa', 'slope_lon_gfa', 'aspect_gfa',
                                                  'vx_gfa', 'vy_gfa'], axis=1)

In [ ]:
glathida_rgis_edited1.to_csv('glathida_rgis_edited1.csv', index=False)